## Environment Setup & Initialization

### Environment Variables
Loads the `.env` file to access the `OPENAI_API_KEY` required for transcription.

### OpenAI Client
Initializes the OpenAI client using the retrieved API key.  
Raises an error if the key is missing to prevent runtime failures.

### Status Output
Prints a confirmation message indicating that the environment is ready and the client is properly configured.


In [5]:
## Cell 1: Environment Control & Imports

import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv(dotenv_path="../.env")

# Initialize the OpenAI client
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("❌ OPENAI_API_KEY not found in the .env file.")

client = OpenAI(api_key=api_key)

print("✅ Environment loaded and OpenAI client initialized.")

✅ Environment loaded and OpenAI client initialized.


## Load Pipeline State & Validate Audio File

### Session State Loading
Reads `session_state.json` to retrieve the `active_video_id` generated during ingestion.  
Ensures Notebook 01 was executed and that the video ID exists.

### Path Construction
Builds the expected paths for:
- The MP3 audio file  
- The TXT transcription output  

Both paths are derived from the active video ID.

### File Validation
Checks that the audio file exists in `../data/`.  
If missing, the pipeline stops and instructs the user to rerun Notebook 01.  
Prints the file size for confirmation.

### Result
Confirms the active video ID and validates that the audio file is ready for transcription.


In [6]:
import json
import os

# Shared pipeline state
STATE_PATH = "../data/session_state.json"

# Load the active video ID from the pipeline state
if not os.path.exists(STATE_PATH):
    raise FileNotFoundError(
        "❌ session_state.json not found. Please run Notebook 01 first."
    )

with open(STATE_PATH, "r", encoding="utf-8") as f:
    session_state = json.load(f)

VIDEO_ID = session_state.get("active_video_id")

if not VIDEO_ID:
    raise ValueError(
        "❌ 'active_video_id' is missing or empty in session_state.json."
    )

print(f"🔄 Active Video ID: {VIDEO_ID}")

# Define input and output paths
AUDIO_PATH = os.path.join("../data", f"{VIDEO_ID}.mp3")
TXT_OUTPUT_PATH = os.path.join("../data", f"{VIDEO_ID}.txt")

# Verify that the audio file exists
if not os.path.exists(AUDIO_PATH):
    raise FileNotFoundError(
        f"❌ Audio file not found: {AUDIO_PATH}. Please run Notebook 01 first."
    )

size_mb = os.path.getsize(AUDIO_PATH) / (1024 * 1024)
print(f"✅ Audio file found: {AUDIO_PATH} ({size_mb:.2f} MB)")

🔄 Active Video ID: 4_oufjP2yeM
✅ Audio file found: ../data\4_oufjP2yeM.mp3 (15.92 MB)


## Whisper Transcription Process

### Timestamp Formatting
Defines a helper function that converts raw seconds into a readable `[MM:SS]` timestamp for each segment.

### Whisper API Call
Opens the MP3 file and sends it to the `whisper-1` model using verbose JSON output with segment‑level timestamps.  
Validates that Whisper returned segments before continuing.

### Transcript Assembly
Iterates through each Whisper segment, formats its start time, and builds a clean text transcript where every line includes:
- A timestamp  
- The spoken text  

Produces a preview of the final transcription.

### Error Handling
Catches and reports any issues during the transcription process, such as missing segments or API failures.


In [7]:
## Cell 3: Transcribe Audio with Whisper

def to_timestamp(seconds: float) -> str:
    """Convert seconds to [MM:SS] format."""
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"[{minutes:02d}:{secs:02d}]"


print("🎙️ Transcribing audio with Whisper-1...")

try:
    with open(AUDIO_PATH, "rb") as audio_file:
        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            language="en",
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )

    if response.segments is None:
        raise ValueError("❌ Whisper did not return any transcription segments.")

    # Build the transcript with timestamps
    transcription_lines = []

    for segment in response.segments:
        timestamp = to_timestamp(segment.start)
        transcription_lines.append(f"{timestamp} {segment.text.strip()}")

    transcription_text = "\n".join(transcription_lines)

    print("✅ Transcription completed successfully!")
    print(f"\n📝 Preview:\n{'-' * 40}")
    print(transcription_text[:350] + "...")
    print(f"{'-' * 40}")

except Exception as e:
    print(f"❌ Whisper transcription failed: {e}")

🎙️ Transcribing audio with Whisper-1...
✅ Transcription completed successfully!

📝 Preview:
----------------------------------------
[00:00] If the ATS can't read your resume, it will be rejected before a human ever sees it.
[00:03] I know because I was the former chief growth officer at WeWork where I read tens of thousands
[00:07] of resumes and personally hired thousands of people.
[00:10] Now I'm the founder of Teal, and I know firsthand how ATS systems actually work.
[00:15...
----------------------------------------


## Save Transcript to File

### Write Output
Saves the full transcription text into a `.txt` file located in `../data/`, using the active video ID as the filename.

### Confirmation
Prints a success message showing the exact path of the generated transcript file.

### Purpose
This step finalizes the transcription stage by persisting the processed text so it can be used later for chunking, embedding, or RAG workflows.


In [8]:
## Cell 4: Save Transcript

with open(TXT_OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(transcription_text)

print("✅ Transcript saved successfully.")
print(f"📄 Output file: {TXT_OUTPUT_PATH}")

✅ Transcript saved successfully.
📄 Output file: ../data\4_oufjP2yeM.txt


# Notebook 2 — Transcription Summary

This notebook handles the full transcription workflow using Whisper.

## 1. Environment Setup
Loads environment variables from `.env` and initializes the OpenAI client.  
Ensures the API key is present before continuing.

## 2. Pipeline State & File Validation
Reads `session_state.json` to recover the active video ID from Notebook 1.  
Builds the paths for the MP3 input and TXT output.  
Verifies that the audio file exists and is ready for transcription.

## 3. Whisper Transcription
Sends the audio file to the `whisper-1` model with segment‑level timestamps.  
Formats each segment into `[MM:SS] text` lines and assembles the full transcript.  
Shows a preview and handles errors gracefully.

## 4. Save Transcript
Writes the final transcription to a `.txt` file in `../data/`, named after the video ID.

## Overall
This notebook transforms the downloaded YouTube audio into a clean, timestamped text transcript, preparing the data for later processing steps such as chunking, embedding, or RAG.
